In [1]:
begin
    using Pkg
    dev_folder = joinpath(@__DIR__, "../Examples")
    Pkg.activate(dev_folder)
end
Threads.nthreads()

  Activating project at `~/Realizibility_index/BindingAndCatalysis.jl/Examples`


24

In [2]:
using Polyhedra
using GLMakie # for plotting, we use Makie backend, could also be GLMakie or WGLMakie...
using Revise
using BindingAndCatalysis # import the package

[ Info: Precompiling BindingAndCatalysis [b532a4b1-2c45-4c12-bcaf-8695372aa40c] (cache misses: include_dependency fsize change (2), incompatible header (4))


A TF dimerize and then binds with a promoter the complex can trigger the production of a protein, and the degradation of the protein can happened in all forms, complex, monomer and dimer, but we know the degradation on complex have to be slower than the degradation of monomer and dimer,  so we can treat the degradation happens only on monomer and dimer.

In [5]:
model = let 
    x = [:D, :P, :P₂, :C]
    q = [:tP, :tD]
    N = [0 2 -1 0
        1 0 1 -1]
    Bnc(N=N, x_sym=x, q_sym=q)
end

----------Binding Network Summary:-------------
Number of species (n): 4
Number of conserved quantities (d): 2
Number of reactions (r): 2
L matrix: sparse([1, 2, 2, 1, 2], [1, 2, 3, 4, 4], [1, 1, 2, 1, 2], 2, 4)
N matrix: sparse([2, 1, 1, 2, 2], [1, 2, 3, 3, 4], [1, 2, -1, 1, -1], 2, 4)
Direction of binding reactions: forward
Catalysis involved: No
Regimes constructed: No
-----------------------------------------------

In [6]:
show_conservation(model)

2-element Vector{Symbolics.Equation}:
 tP ~ C + D
 tD ~ 2C + P + 2P₂

In [11]:
Vector{Symbol} <: Union{<:AbstractVector,Nothing}

true

In [7]:
using SparseArrays

In [8]:
let
    Γ = [1 -1 -1]
    Π = [1 0 0
        0 1 0
        0 0 1]
    k_sym = [:k₁, :k₂, :k₃]
    q_picked=[:tP]
    x_picked=[:C, :P, :P₂]
    update_catalysis!(model;Γ=Γ, Π=Π, k_sym=k_sym, x_picked=x_picked, q_picked=q_picked)
end


[ Info: q is reordered to make catalysis-involving species first


In [10]:
model.catalysis.r_v

1

In [11]:
P* pi *H

1-element Vector{Int64}:
 -1

In [12]:
model = let 
    L = [0 1 1; 1 0 1]
    N = [1 1 -1]
    Bnc(L=L,N=N)
end

----------Binding Network Summary:-------------
Number of species (n): 3
Number of conserved quantities (d): 2
Number of reactions (r): 1
L matrix: [0 1 1; 1 0 1]
N matrix: [1 1 -1]
Direction of binding reactions: forward
Catalysis involved: No
Regimes constructed: No
-----------------------------------------------

In [19]:
get_H(model,2)

3×3 SparseMatrixCSC{Float64, Int64} with 5 stored entries:
 -1.0  1.0  1.0
  1.0   ⋅    ⋅ 
   ⋅   1.0   ⋅ 

In [23]:
M1= get_M(model,2)

3×3 SparseMatrixCSC{Int64, Int64} with 5 stored entries:
 ⋅  1   ⋅
 ⋅  ⋅   1
 1  1  -1

In [24]:
M2 = get_M(model,4)

3×3 SparseMatrixCSC{Int64, Int64} with 5 stored entries:
 ⋅  ⋅   1
 ⋅  ⋅   1
 1  1  -1

In [22]:
a = inv([0 0 1 ;1 1 -1;0 -1 1])

3×3 Matrix{Float64}:
 -0.0  1.0   1.0
  1.0  0.0  -1.0
  1.0  0.0   0.0

In [25]:
M1 *a

3×3 Matrix{Float64}:
 1.0  0.0  -1.0
 1.0  0.0   0.0
 0.0  1.0   0.0

In [26]:
M2 *a

3×3 Matrix{Float64}:
 1.0  0.0  0.0
 1.0  0.0  0.0
 0.0  1.0  0.0

In [ ]:
function S_to_S_pos_neg(S::Matrix)
    S_p = zeros(size(S))
    S_n = zeros(size(S))
    for i in eachindex(S)
        if S[i] > 0
            S_p[i] = S[i]
        elseif S[i] < 0
            S_n[i] = -S[i]
        end
    end
    return S_p, S_n
end
    